In [350]:
from pynq import Overlay, DefaultIP, MMIO, allocate 
import numpy as np
from enum import Enum


import struct

bram = MMIO(BRAM_ADDR,ADDRESS_RANGE)
BRAM_ADDR = 0x80010000
NUM_TAPS_OFFSET = 0x0
LOWER_CUTOFF_OFFSET = 0x4
UPPER_CUTOFF_OFFSET = 0x8
SAMPLING_RATE_OFFSET = 0xC
READ_SUCCESS = 0x10

ADDRESS_RANGE = 0x1000
ADDRESS_OFFSET = 0x00
def hex(val):
    return f"0x{val:08X}"

class AddDriver(DefaultIP):
    bindto = ['xilinx.com:hls:firTop:1.0']

    def process(self):
        self.write(0x10, BRAM_ADDR)
        print(hex(self.read(0x10)))

#load overlay
overlay = Overlay('Motor_Simulator_wrapper.bit')
add_ip = overlay.firTop_0

add_ip.process()

#add dma
dma = overlay.axi_dma 
dma_send = overlay.axi_dma.sendchannel 
dma_recv = overlay.axi_dma.recvchannel 

def set_bram_data(numTaps, lowerCutoff, upperCutoff, samplingRate):
    bram.write(NUM_TAPS_OFFSET, numTaps)
    bram.write(LOWER_CUTOFF_OFFSET, lowerCutoff)
    bram.write(UPPER_CUTOFF_OFFSET, upperCutoff)
    bram.write(SAMPLING_RATE_OFFSET, samplingRate)
    bram.write(READ_SUCCESS, 0)

def isFirReadingMem():
    return bram.read(READ_SUCCESS)


def generate_sine_samples(freq=100, sample_rate=100000, num_samples=100):
    t = np.linspace(0, num_samples/sample_rate, num_samples)
    sine = np.sin(2 * np.pi * freq * t)
    return (sine * 32767).astype(np.int32)

def envelope_to_pwm(data, pwm_max=255):
    data = np.array(data)
    max_val = np.max(data)
    pwm = (data / max_val * pwm_max).astype(np.int32)
    return pwm

def stream(integerList):
    CONTROL_REGISTER = 0x0 
    data_size = 1000
    input_buffer = allocate(shape=(data_size,), dtype=np.uint32) 
    for i in range(len(integerList)): 
        input_buffer[i] = integerList[i]
    dma_send.transfer(input_buffer) 
    output_buffer = allocate(shape=(data_size,), dtype=np.uint32) 
    dma_recv.transfer(output_buffer)
    for i in range(len(integerList)): 
        integerList[i] = output_buffer[i] 
    del input_buffer, output_buffer 
    return integerList


0x80010000


In [411]:
set_bram_data(100, 2000, 5000, 100000)
isFirReadingMem()


0

In [412]:
sample = generate_sine_samples(1, 100000,100)
#print(sample)


In [415]:
stream(sample)
#pwm = envelope_to_pwm(sample)
#print(f"Peak PWM value : {np.max(pwm)}")   # 255
#print(f"Min  PWM value : {np.min(pwm)}")   # 0
#print(f"PWM values     : {pwm}")

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], dtype=int32)

In [328]:
overlay.ip_dict

{'axi_dma': {'type': 'xilinx.com:ip:axi_dma:7.1',
  'mem_id': 'S_AXI_LITE',
  'memtype': 'REGISTER',
  'gpio': {},
  'interrupts': {},
  'parameters': {'C_DLYTMR_RESOLUTION': '125',
   'C_ENABLE_MULTI_CHANNEL': '0',
   'C_FAMILY': 'zynquplus',
   'C_INCLUDE_MM2S': '1',
   'C_INCLUDE_MM2S_DRE': '0',
   'C_INCLUDE_MM2S_SF': '1',
   'C_INCLUDE_S2MM': '1',
   'C_INCLUDE_S2MM_DRE': '0',
   'C_INCLUDE_S2MM_SF': '1',
   'C_INCLUDE_SG': '0',
   'C_INCREASE_THROUGHPUT': '0',
   'C_MICRO_DMA': '0',
   'C_MM2S_BURST_SIZE': '16',
   'C_M_AXIS_MM2S_CNTRL_TDATA_WIDTH': '32',
   'C_M_AXIS_MM2S_TDATA_WIDTH': '32',
   'C_M_AXI_MM2S_ADDR_WIDTH': '64',
   'C_M_AXI_MM2S_DATA_WIDTH': '64',
   'C_M_AXI_S2MM_ADDR_WIDTH': '64',
   'C_M_AXI_S2MM_DATA_WIDTH': '32',
   'C_M_AXI_SG_ADDR_WIDTH': '64',
   'C_M_AXI_SG_DATA_WIDTH': '32',
   'C_NUM_MM2S_CHANNELS': '1',
   'C_NUM_S2MM_CHANNELS': '1',
   'C_PRMRY_IS_ACLK_ASYNC': '0',
   'C_S2MM_BURST_SIZE': '16',
   'C_SG_INCLUDE_STSCNTRL_STRM': '0',
   'C_SG_LENGTH_WID

In [4]:
print(overlay.ip_dict.keys())

dict_keys(['axi_dma', 'firTop_0', 'zynq_ultra_ps_e_0'])
